# v8: Dynamic Coalition Network

Seeds select top-k nodes, then recruit teammates through a **learned edge matrix**.
Each edge is a free parameter — the network learns which nodes should co-fire.
3,296,616 params.

In [ ]:
# ===== Setup =====
import os, sys, subprocess

REPO_URL  = 'https://github.com/tanushappapogu-max/Dynamic-Coalition-Network-DCN-.git'
REPO_DIR  = 'Dynamic-Coalition-Network-DCN-'
BRANCH    = 'main'
EXPECTED_PARAMS = 3_296_616

IN_COLAB = 'COLAB_GPU' in os.environ or 'google.colab' in str(globals().get('get_ipython', lambda: ''))

if IN_COLAB:
    !pip install -q torch pyyaml matplotlib seaborn networkx scikit-learn numpy

    while os.path.basename(os.getcwd()) == REPO_DIR:
        os.chdir('..')

    !rm -rf {REPO_DIR}
    !git clone --branch {BRANCH} --single-branch {REPO_URL}
    os.chdir(REPO_DIR)

if '.' not in sys.path:
    sys.path.insert(0, '.')

head = subprocess.run(['git', 'log', '-1', '--pretty=%h %s'],
                      capture_output=True, text=True).stdout.strip()
print(f'repo   : {os.getcwd()}')
print(f'commit : {head}')

import torch, yaml
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch  : {torch.__version__} | device: {device}')
if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)} | '
          f'{torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

with open('configs/experiment.yaml') as f:
    config = yaml.safe_load(f)

from src.models.coalition import CoalitionModel, CoalitionGraphFFN
_m = CoalitionModel(config)
_n = _m.count_parameters()
assert hasattr(CoalitionGraphFFN, '_get_routing_params'), 'wrong module'
assert _n == EXPECTED_PARAMS, (
    f'WRONG VERSION: {_n:,} params, expected {EXPECTED_PARAMS:,}. '
    f'Runtime > Restart, then re-run.')
# Verify edge_logits exist (v8 design)
assert any(hasattr(l.ffn, 'edge_logits') for l in _m.layers), (
    'No edge_logits found — this is not the v8 edge-matrix design.')
print(f'version: v8 OK ({_n:,} params) | d_model={config["model"]["d_model"]} '
      f'n_nodes={config["coalition"]["n_nodes"]} n_seeds={config["coalition"]["n_seeds"]}')

In [ ]:
# ===== Generate Data =====
from torch.utils.data import DataLoader
from src.data.arithmetic import create_datasets, classify_expression

print('Generating datasets...')
datasets = create_datasets(config)
for name, ds in datasets.items():
    print(f'  {name}: {len(ds)} examples')

print('\nSample expressions:')
for i in range(8):
    expr, result = datasets['train'].data[i]
    print(f'  {expr} = {result}  [{classify_expression(expr)}]')

bs = config['training']['batch_size']
train_loader = DataLoader(datasets['train'], batch_size=bs, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(datasets['val'], batch_size=bs, num_workers=2, pin_memory=True)
test_loader = DataLoader(datasets['test'], batch_size=bs, num_workers=2, pin_memory=True)
gen_loader = DataLoader(datasets['gen_test'], batch_size=bs, num_workers=2, pin_memory=True)

## v8 — Does a learned edge matrix make recruitment beat top-k?

3 conditions x 3 seeds = 9 runs. `v8_recruit` vs `C_matchedk` is the number that matters.

In [ ]:
# ===== v8: Edge-Matrix Recruitment Test =====
# 3 conditions x 3 seeds = 9 runs.
#
#   --seeds 3   : cheap config (d160/3L/25K) ~1 hr on T4
#   --full      : real config (d256/4L/80K) ~4 hr on T4
#
# The verdict is at the bottom: v8_recruit vs C_matchedk.

!python edge_recruitment_test.py --seeds 3

In [ ]:
# ===== Specialization Analysis: Which nodes light up for which tasks? =====

from src.evaluation.metrics import collect_activation_patterns
from src.evaluation.visualize import plot_activation_heatmap, compute_cluster_purity

print('Collecting activation patterns from 1000 test examples...')
patterns = collect_activation_patterns(model, datasets['test'], device, n_samples=1000)

from collections import Counter
type_counts = Counter(p['expr_type'] for p in patterns)
print(f'\nExpression types in sample:')
for t, c in type_counts.most_common():
    print(f'  {t}: {c}')

for layer_idx in range(config['model']['n_layers']):
    plot_activation_heatmap(
        patterns, layer_idx=layer_idx,
        save_path=f'results/coalition/heatmap_layer{layer_idx+1}.png'
    )

from IPython.display import Image, display
for layer_idx in range(config['model']['n_layers']):
    print(f'\n--- Layer {layer_idx+1} ---')
    display(Image(filename=f'results/coalition/heatmap_layer{layer_idx+1}.png'))

In [ ]:
# ===== Learned Edge Matrix: What co-firing structure emerged? =====

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

for layer_idx, layer in enumerate(model.layers):
    if hasattr(layer.ffn, 'edge_logits'):
        ew = torch.sigmoid(layer.ffn.edge_logits).detach().cpu().numpy()
        mask = 1.0 - np.eye(ew.shape[0])
        vals = ew[mask.astype(bool)]

        print(f'--- Layer {layer_idx+1} Edge Weights ---')
        print(f'  mean={vals.mean():.3f}  std={vals.std():.3f}')
        print(f'  strong (>0.8): {(vals > 0.8).sum()}')
        print(f'  weak   (<0.2): {(vals < 0.2).sum()}')

        fig, ax = plt.subplots(figsize=(8, 6))
        im = ax.imshow(ew * mask, cmap='viridis', vmin=0, vmax=1)
        ax.set_title(f'Layer {layer_idx+1} Edge Weights (learned)')
        ax.set_xlabel('Recruited node'); ax.set_ylabel('Seed node')
        plt.colorbar(im, ax=ax)
        plt.tight_layout()
        plt.savefig(f'results/coalition/edges_layer{layer_idx+1}.png', dpi=150)
        plt.show()

        # Top recruitment pairs
        pairs = []
        n = ew.shape[0]
        for i in range(n):
            for j in range(n):
                if i != j:
                    pairs.append((i, j, ew[i, j]))
        pairs.sort(key=lambda x: -x[2])
        print(f'\n  Strongest edges (seed → recruit):')
        for i, j, w in pairs[:10]:
            print(f'    Node {i} → Node {j}: {w:.3f}')
        break

In [ ]:
# ===== Graph Visualization: Edge matrix as a network =====

import networkx as nx
import matplotlib.pyplot as plt
from IPython.display import Image, display

for layer in model.layers:
    if hasattr(layer.ffn, 'edge_logits'):
        ew = torch.sigmoid(layer.ffn.edge_logits).detach().cpu().numpy()
        break

G = nx.DiGraph()
n = ew.shape[0]
for i in range(n):
    G.add_node(i)
for i in range(n):
    for j in range(n):
        if i != j and ew[i, j] > 0.6:
            G.add_edge(i, j, weight=ew[i, j])

pos = nx.spring_layout(G, seed=42, k=2)
fig, ax = plt.subplots(figsize=(10, 8))

edges = G.edges(data=True)
weights = [e[2]['weight'] for e in edges]
nx.draw_networkx_nodes(G, pos, node_size=500, node_color='#3498db', ax=ax)
nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold', ax=ax)
nx.draw_networkx_edges(G, pos, width=[w * 3 for w in weights],
                       alpha=[w * 0.8 for w in weights],
                       edge_color='#e74c3c', arrows=True, ax=ax)
ax.set_title(f'Learned recruitment graph (edges > 0.6)')
plt.tight_layout()
plt.savefig('results/coalition/graph_structure.png', dpi=150)
plt.show()
print(f'{len(edges)} edges above threshold')

In [ ]:
# ===== Training Curves =====

import json
import matplotlib.pyplot as plt

with open('results/coalition/training_log.json') as f:
    log = json.load(f)

epochs = [e['epoch'] for e in log]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0,0].plot(epochs, [e['train_task_loss'] for e in log], label='Train')
axes[0,0].plot(epochs, [e['val_loss'] for e in log], label='Val')
axes[0,0].set_title('Task Loss'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(epochs, [e['val_accuracy'] for e in log])
axes[0,1].set_title('Validation Accuracy'); axes[0,1].grid(True, alpha=0.3)

axes[0,2].plot(epochs, [e['temperature'] for e in log])
axes[0,2].set_title('Temperature'); axes[0,2].grid(True, alpha=0.3)

axes[1,0].plot(epochs, [e['coalition_size_mean'] for e in log])
axes[1,0].set_title('Avg Coalition Size'); axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(epochs, [e['node_freq_std'] for e in log])
axes[1,1].set_title('Node Frequency Std (specialization)'); axes[1,1].grid(True, alpha=0.3)

axes[1,2].plot(epochs, [e.get('n_close_pairs', 0) for e in log])
axes[1,2].set_title('Close Node Pairs (sim > 0.7)'); axes[1,2].grid(True, alpha=0.3)

for ax in axes.flat:
    ax.set_xlabel('Epoch')

plt.tight_layout()
plt.savefig('results/coalition/training_dashboard.png', dpi=150)
plt.show()
print('\nKey: Does node_freq_std increase? Do close_pairs form? That means self-organization is happening.')

In [ ]:
# ===== Comparison Table + Training Curves Overlay =====

import json
import torch
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load all three results
models = {}
for name in ['coalition', 'dense', 'moe']:
    path = f'results/{name}/final.pt'
    try:
        ckpt = torch.load(path, map_location=device, weights_only=False)
        models[name] = ckpt
        print(f'Loaded {name}: {ckpt["n_params"]:,} params')
    except FileNotFoundError:
        print(f'WARNING: {path} not found — skipping {name}')

# --- Comparison Table ---
print('\n' + '=' * 70)
print('FINAL COMPARISON')
print('=' * 70)
print(f'{"Model":<12} {"Params":>10} {"TestAcc":>10} {"GenAcc":>10} {"Speed(ms)":>12} {"Specialization":>15}')
print('-' * 70)

for name in ['dense', 'moe', 'coalition']:
    if name not in models:
        continue
    m = models[name]
    spec = '—'
    if name == 'coalition' and 'final_diagnostics' in m:
        diag = m['final_diagnostics']
        freq_std = diag.get('node_freq_std', 0)
        spec = f'std={freq_std:.3f}'
    print(
        f'{name:<12} '
        f'{m["n_params"]:>10,} '
        f'{m["test_metrics"]["accuracy"]:>10.4f} '
        f'{m["gen_metrics"]["accuracy"]:>10.4f} '
        f'{m["inference_ms"]:>12.3f} '
        f'{spec:>15}'
    )

# --- Overlaid Training Curves ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = {'coalition': '#e74c3c', 'dense': '#3498db', 'moe': '#2ecc71'}

for name in ['coalition', 'dense', 'moe']:
    log_path = f'results/{name}/training_log.json'
    try:
        with open(log_path) as f:
            log = json.load(f)
    except FileNotFoundError:
        continue

    epochs = [e['epoch'] for e in log]
    c = colors[name]

    axes[0].plot(epochs, [e['train_task_loss'] for e in log], color=c, label=name, linewidth=2)
    axes[1].plot(epochs, [e['val_loss'] for e in log], color=c, label=name, linewidth=2)
    axes[2].plot(epochs, [e['val_accuracy'] for e in log], color=c, label=name, linewidth=2)

titles = ['Train Loss', 'Val Loss', 'Val Accuracy']
for ax, title in zip(axes, titles):
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/comparison_curves.png', dpi=150)
plt.show()

print('\nDone. Results saved to results/comparison_curves.png')